# What are its radius and mass? Fit the transits and the velocities together

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/tess_02_radius_and_mass.ipynb)

In the bootcamp you measured a planet's radius from its dip and its mass from its star's wobble, separately, with simple models. Here you fit one model to both at once: up to eight TESS sectors and the survey's velocities share the same orbit. Velocities taken during a transit are left out, as in notebook 4. It also fits the full shape of the transit, so the radius no longer comes out too big or too small.

| Part | What you do |
|---|---|
| 1 | Fit the transits and velocities together, and compare with the published radius and mass |
| 2 | Re-run with another catalogue's values for the star, and see what moves |

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. Each part ends with a cell that puts your answers into a single table, which is what you hand in. Each fit takes a few minutes.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("batman") is None:
    %pip install -q "habtrain[tess] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/tess_02_radius_and_mass.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Part 1: Fit the transits and velocities together

**What the model adds.** Three things the bootcamp's simple steps left out:
- **The impact parameter:** how far from the middle of the star the planet crosses, from 0 (straight across the middle) to 1 (grazing the edge). A planet crossing near the edge makes a shorter, shallower dip.
- **Limb darkening:** the star's edge is dimmer than its middle, so the dip has rounded shoulders rather than a flat bottom.
- **The star's mass and radius,** taken from published values with their uncertainties. Together with the period they set how fast the planet crosses, which ties the transit's length to the impact parameter.

**Times.** TESS times are BTJD, which is the full Barycentric Julian Date minus 2457000. The survey's velocities come as full BJD. The notebook converts everything to BTJD; mixing the two silently ruins a fit.

In [ ]:
# Every velocity the survey released, once, for both parts.
rvs = ht.data.load_rvs()

# Worked example: TOI-669 b. Its orbit, transit length and star (mass and radius with uncertainties).
orb = ht.data.orbit("TOI-669 b")
hours = ht.data.transit_duration("TOI-669 b")
star = ht.data.star("TOI-669 b")

# Download up to 8 TESS sectors, flatten each with its transits hidden, and keep the data near transit.
prep = ht.joint.prepare("TOI-669 b", rvs, orb, hours)
print(f"Sectors {prep.sectors}: {len(prep.t_lc)} light-curve points near transit, {len(prep.rv)} velocities "
      f"({prep.rv_in_transit} taken during a transit are left out, as in notebook 4)")

In [ ]:
# Fit transit and velocities together; it prints how long it took. The star's values enter as priors.
res = ht.joint.fit(prep, star["mass"], star["radius"])
show = lambda v, unit: f"{v[0]:.2f} (+{v[1]:.2f}, -{v[2]:.2f}) {unit}"
print("Radius:", show(res.radius, "Earth radii"))
print("Mass:  ", show(res.mass, "Earth masses"))
print("K:     ", show(res.K, "m/s"), "  Impact parameter:", show(res.impact, ""))
print(f"Converged: {res.converged} (autocorrelation {res.autocorr_steps:.0f} steps, run {res.steps}). The fit took {res.seconds / 60:.0f} minutes.")

# The fitted transit and velocity curves through the data.
fig = ht.plot.joint(ht.joint.curves(prep, res), "TOI-669 b: one orbit, two kinds of data")
plt.show()

The published values for TOI-669 b are a radius of 2.586 (+0.13, -0.11) Earth radii and a mass of 10.0 ± 1.4 Earth masses ([Polanski et al. 2024](https://arxiv.org/abs/2405.14786), with the radius from the transit fits of [MacDougall et al. 2023](https://doi.org/10.3847/1538-3881/acd557)). The fit lands on both.

### Your planet

Fit the planet you vetted in the previous notebook. Choose it from the map.

In [ ]:
# The planets you can fit, and a menu that circles your choice on the map. TOI-669 b, the worked example, is left out.
planets = ht.data.planets()
planets = planets[planets.planet != "TOI-669 b"].reset_index(drop=True)
fig = ht.plot.planet_map(planets)
plt.show()
chooser = ht.choose.planet(planets, ht.plot.planet_map)

> **Write down: your planet's radius and mass**
>
> - Choose your planet in the menu above. If the menu does not appear, put its number from the map in `MY_PICK`.
> - Run the fit. The cell prints the radius, mass, K and impact parameter, and whether the fit converged.
> - In one or two lines: did it converge, and does anything in the plots look wrong? Put it in `LOOKS_RIGHT`.
>
> **Where to look:** the printed results and the two plots.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_PICK = ...       # only if the menu did not appear: your planet's number on the map
LOOKS_RIGHT = ...   # one or two lines, in quotes

# Your planet: what you chose in the menu, else the number you typed, else a random one.
planet = ht.choose.resolve(chooser, MY_PICK, planets, "planet", "planet")
star = ht.data.star(planet)

# Prepare and fit, as in the worked example (it takes about as long).
mine = ht.joint.prepare(planet, rvs, ht.data.orbit(planet), ht.data.transit_duration(planet))
my_res = ht.joint.fit(mine, star["mass"], star["radius"])
print(f"Sectors {mine.sectors}; {len(mine.rv)} velocities ({mine.rv_in_transit} taken during a transit are left out)")
print("Radius:", show(my_res.radius, "Earth radii"), "| Mass:", show(my_res.mass, "Earth masses"))
print("Impact parameter:", show(my_res.impact, ""), f"| Converged: {my_res.converged}")
if not my_res.converged:
    print('Not converged: change the fit line above to my_res = ht.joint.fit(mine, star["mass"], star["radius"], steps=40000) and run this cell again.')
my_fig = ht.plot.joint(ht.joint.curves(mine, my_res), f"{planet}: one orbit, two kinds of data")
plt.show()

# Keep your results for the hand-in table.
ht.record.save("my_planet", planet, "your planet")
ht.record.save("radius", show(my_res.radius, "Earth radii"), "radius")
ht.record.save("mass", show(my_res.mass, "Earth masses"), "mass")
ht.record.save("impact", show(my_res.impact, ""), "impact parameter")
ht.record.save("converged", str(my_res.converged), "whether the fit converged")
ht.record.save("looks_right", LOOKS_RIGHT, "whether the fit looks right")

### Hand in Part 1

Run the cell below. It shows your Part 1 answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `joint-fit`, and download them.

Once you have handed in, the step shows the answer key: the published radius and mass of every planet on the map.

In [ ]:
# Your Part 1 answers and the fit plot, saved as your hand-in.
ht.record.handin("joint-fit", ["my_planet", "radius", "mass", "impact", "converged", "looks_right"], figures={"fit": my_fig})

## Part 2: Does the answer depend on what you assume?

The radius depends on the star's radius, and both depend on what was assumed. So far the fit used the star's mass and radius from the NASA Exoplanet Archive, which for these stars come from the published papers, where the survey measured each star from its own spectra ([MacDougall et al. 2023](https://doi.org/10.3847/1538-3881/acd557)). Here you swap them for the TESS Input Catalog's values (TIC v8.2), which come from the star's colours and its Gaia distance, with no spectrum ([Stassun et al. 2019](https://arxiv.org/abs/1905.10694)), and fit again. In a new session, run Part 1 first.

**Predict first**

Will your planet's radius move by more or less than its own uncertainty?

In [ ]:
# Your prediction, before the second fit.
MY_PREDICTION = ...   # "more", "less" or "unsure"
ht.record.predict("star_values", MY_PREDICTION)

In [ ]:
# This only runs once you have made your prediction above.
if "mine" not in dir():
    raise SystemExit("Run Part 1 first: Part 2 refits the planet you fitted there.")
if ht.record.revealed("star_values"):
    # The star's mass and radius from the TESS Input Catalog, then the same fit again.
    tic = ht.vet.tic_star(star["tic"])
    print(f"Archive star: mass {star['mass'][0]:.3f} ± {star['mass'][1]:.3f}, radius {star['radius'][0]:.3f} ± {star['radius'][1]:.3f}")
    print(f"TIC star:     mass {tic['mass'][0]:.3f} ± {tic['mass'][1]:.3f}, radius {tic['radius'][0]:.3f} ± {tic['radius'][1]:.3f}")
    res_tic = ht.joint.fit(mine, tic["mass"], tic["radius"])
    print("Radius with archive star:", show(my_res.radius, "Earth radii"))
    print("Radius with TIC star:    ", show(res_tic.radius, "Earth radii"))
    print("Mass with TIC star:      ", show(res_tic.mass, "Earth masses"))

> **Write down: what moves when the star changes**
>
> - How far did the radius move, compared with its uncertainty? Put it in `RADIUS_MOVED`.
> - Which star mass and radius would you trust more, the archive's or the TIC's, and why? One or two lines. Put it in `TRUST`.
>
> **Where to look:** the two fits printed above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
RADIUS_MOVED = ...   # one line, in quotes
TRUST = ...          # one or two lines, in quotes

# Keep the answers for your hand-in table.
ht.record.save("radius_moved", RADIUS_MOVED, "how far the radius moved with the TIC star")
ht.record.save("trust", TRUST, "which value you would trust, and why")

**Worth knowing**

The star is one assumption; the orbit's shape is another. This model assumes a circular orbit. A planet whose transit is shorter than a circular orbit allows has to cross near the star's edge in this model, and that can make it look bigger. Fitting an eccentric orbit is beyond this notebook. If you think your planet needs one, say so in your hand-in and bring it to the office hour.

### Hand in Part 2

Run the cell below, check the table, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `star-assumption`, and download them.

In [ ]:
# Your Part 2 answers, saved as your hand-in.
ht.record.handin("star-assumption", ["radius_moved", "trust"])